# Repeatability Analysis (thesis-grade)

Loads the most recent `repeatability` run and reports ISO 9283 pose repeatability (`RP_yz`) at a single goal pose, with separation between **within-visit** repeatability (motor + apriltag detection noise floor) and **between-visit** repeatability (true arm repeatability).

This is the **clean repeatability number** for the thesis: frame-independent, no URDF bias, directly comparable to ISO 9283 specs from commercial arms.

**Within-visit** (samples within one Continue gate):
- Captures motor settling residual + camera-frame noise + apriltag PnP noise.
- Sub-mm here on a stable rig is normal.

**Between-visit** (cluster of per-cycle means):
- Captures actual arm repeatability — gear backlash, motor zero drift, etc.
- This is the headline number you'd quote against UR5/UR10/Niryo specs.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from volcaniarm_calibration.analysis import (
    accuracy_iso9283, align_fk_to_tag, latest_run, list_runs, load_run,
    repeatability_iso9283, summary, threshold_color, threshold_zone,
    WEEDING_ACCEPTABLE_MM, WEEDING_MARGINAL_MM,
)

TEST_NAME = 'repeatability'
RUN_DIR = latest_run(TEST_NAME)
run = load_run(RUN_DIR)
config, tag, fk = run['config'], run['tag'], run['fk']
print(f'run_id:           {config.get("run_id")}')
print(f'status:           {config.get("status")}')
print(f'iterations:       {config["num_cycles"]}')
print(f'samples per visit:{config["samples_per_visit"]}')
print(f'goal (y, z):      {config["goals"][0]}')

target = tag[tag['phase'] == 'target'].copy()

## Within-visit repeatability (per cycle)

For each cycle, compute `RP_yz` over the samples taken within that single visit. Small here = motor + camera noise floor.

In [ ]:
rows = []
for cy in sorted(target['cycle'].unique()):
    sub = target[target['cycle'] == cy]
    rep = repeatability_iso9283(sub, dims=('y', 'z'))
    rows.append({
        'cycle': cy,
        'n': rep['n'],
        'RP_mm': rep['RP_m'] * 1000,
        'worst_mm': rep['worst_m'] * 1000,
        'centroid_y_mm': rep['centroid'][0] * 1000,
        'centroid_z_mm': rep['centroid'][1] * 1000,
    })
per_cycle = pd.DataFrame(rows)
print(per_cycle.to_string(index=False))
if len(per_cycle) > 0:
    rp_within = summary(per_cycle['RP_mm'] / 1000)
    print(f'\nwithin-visit RP (motor + detection noise): {rp_within.in_mm()}')
    worst_within = per_cycle['RP_mm'].max()
    print(f'worst within-visit RP:  {worst_within:.3f} mm   '
          f'(zone: {threshold_zone(worst_within)})')

## Between-visit repeatability (cluster of cycle means)

Take the centroid of each cycle's samples, then compute `RP_yz` over those centroids. **This is the true arm repeatability** — the within-visit noise has averaged out and what remains is the cycle-to-cycle scatter (motor zero drift, gear backlash, control-loop residuals).

In [ ]:
if len(per_cycle) >= 2:
    centroids = per_cycle[['centroid_y_mm', 'centroid_z_mm']].copy()
    centroids.columns = ['y', 'z']
    centroids = centroids / 1000  # back to metres for the metric helpers
    rep_between = repeatability_iso9283(centroids, dims=('y', 'z'))
    print(f'between-visit (cycle centroids):')
    print(f'  n cycles:       {rep_between["n"]}')
    print(f'  RP_yz:          {rep_between["RP_m"]*1000:.3f} mm  '
          f'(zone: {threshold_zone(rep_between["RP_m"]*1000)})')
    print(f'  worst_yz:       {rep_between["worst_m"]*1000:.3f} mm')
    print(f'  centroid (y,z): ({rep_between["centroid"][0]*1000:.3f}, '
          f'{rep_between["centroid"][1]*1000:.3f}) mm')
    s = summary(rep_between['d_to_centroid'])
    print(f'  d-to-centroid:  {s.in_mm()}')
    print('\nThis is the headline thesis number for arm repeatability.')
else:
    print('Need ≥ 2 cycles for between-visit repeatability. Re-run with iterations ≥ 3.')

## All-samples repeatability (lumped)

Single `RP_yz` across every sample in the run, ignoring the cycle structure. Useful for a single quick metric — somewhere between within and between visit, depending on how cycle scatter compares to within-visit scatter.

In [ ]:
rep_all = repeatability_iso9283(target, dims=('y', 'z'))
print(f'all-samples lumped:')
print(f'  n:        {rep_all["n"]}')
print(f'  RP_yz:    {rep_all["RP_m"]*1000:.3f} mm  '
      f'(zone: {threshold_zone(rep_all["RP_m"]*1000)})')
print(f'  worst:    {rep_all["worst_m"]*1000:.3f} mm')

## Per-cycle scatter visualisation

Cycle centroids overlaid with their per-cycle samples, plus the application threshold rings.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

cmap = plt.get_cmap('tab10')
for i, cy in enumerate(sorted(target['cycle'].unique())):
    sub = target[target['cycle'] == cy]
    ax.scatter(sub['y'], sub['z'], marker='x', s=70,
               c=[cmap(i)], label=f'cycle {cy} (n={len(sub)})', zorder=3)
    ax.scatter([sub['y'].mean()], [sub['z'].mean()], marker='o', s=120,
               facecolors='none', edgecolors=cmap(i), linewidths=2,
               zorder=4)

if len(per_cycle) >= 2:
    cy_g = (per_cycle['centroid_y_mm'].mean()) / 1000
    cz_g = (per_cycle['centroid_z_mm'].mean()) / 1000
    ax.scatter([cy_g], [cz_g], marker='*', s=300, c='black',
               label='global centroid', zorder=5)
    theta = np.linspace(0, 2 * np.pi, 200)
    rp = rep_between['RP_m']
    ax.plot(cy_g + rp * np.cos(theta), cz_g + rp * np.sin(theta),
            color=threshold_color(rp * 1000), linewidth=2,
            label=f'between-visit RP_yz = {rp*1000:.2f} mm')
    for r_mm, style, lbl in (
        (WEEDING_ACCEPTABLE_MM, ':', f'acceptable ({WEEDING_ACCEPTABLE_MM:.0f} mm)'),
        (WEEDING_MARGINAL_MM, '--', f'marginal ({WEEDING_MARGINAL_MM:.0f} mm)'),
    ):
        r = r_mm / 1000
        ax.plot(cy_g + r * np.cos(theta), cz_g + r * np.sin(theta),
                color='gray', linestyle=style, linewidth=1, label=lbl)

gy, gz = config['goals'][0]
ax.scatter([gy], [gz], marker='+', s=200, c='red', label='goal', zorder=5)

ax.set_xlabel('y [m]')
ax.set_ylabel('z [m]')
ax.set_title(f'repeatability  ({config["run_id"]})')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
ax.legend(loc='best', fontsize=8)
fig.tight_layout()

## Box plot of per-cycle distance-to-centroid

One box per cycle. Narrow box = good motor settling within that visit. Box-to-box differences = how cycles drift from each other.

In [ ]:
if rep_all['n'] >= 2:
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    cycles = sorted(target['cycle'].unique())
    data = []
    for cy in cycles:
        sub = target[target['cycle'] == cy]
        rep = repeatability_iso9283(sub, dims=('y', 'z'))
        data.append((rep['d_to_centroid'].to_numpy() * 1000) if rep['n'] >= 2
                    else np.array([0.0]))
    ax.boxplot(data, labels=[f'c{c}' for c in cycles])
    ax.axhline(WEEDING_ACCEPTABLE_MM, color='gray', linestyle=':', alpha=0.6,
               label=f'acceptable ({WEEDING_ACCEPTABLE_MM:.0f} mm)')
    ax.axhline(WEEDING_MARGINAL_MM, color='gray', linestyle='--', alpha=0.6,
               label=f'marginal ({WEEDING_MARGINAL_MM:.0f} mm)')
    ax.set_xlabel('cycle')
    ax.set_ylabel('distance to per-cycle centroid [mm]')
    ax.set_title('within-visit scatter per cycle')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
else:
    print('Need ≥ 2 samples per visit + ≥ 2 cycles for the box plot.')